In [ ]:
# TensorFlow for deep learning
import tensorflow as tf

# Numerical operations
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# File handling
import os

# Image preprocessing
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Pretrained EfficientNet model
from tensorflow.keras.applications import EfficientNetB0

# Layers and model building
from tensorflow.keras import layers, models

# Evaluation metrics
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

In [ ]:
# Define three-class dataset location
dataset_path = "../dataset_threeclass"

# Display folders
print(os.listdir(dataset_path))

In [ ]:
# Define image size
IMG_SIZE = 224

# Define batch size
BATCH_SIZE = 32

# Create image generator
datagen = ImageDataGenerator(

    # Normalize images
    rescale=1./255,

    # Validation split
    validation_split=0.2
)

# Load training dataset
train_data = datagen.flow_from_directory(

    dataset_path,

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode='categorical',

    subset='training'
)

# Load validation dataset
val_data = datagen.flow_from_directory(

    dataset_path,

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode='categorical',

    subset='validation'
)

In [ ]:
# Display class labels
print(train_data.class_indices)

In [ ]:
# Load pretrained EfficientNetB0
base_model = EfficientNetB0(

    weights='imagenet',

    include_top=False,

    input_shape=(224,224,3)
)

# Freeze pretrained layers
base_model.trainable = False

# Build classification model
model = models.Sequential([

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation='relu'),

    layers.Dropout(0.5),

    # Three output classes
    layers.Dense(3, activation='softmax')
])

In [ ]:
# Compile model
model.compile(

    optimizer='adam',

    loss='categorical_crossentropy',

    metrics=['accuracy']
)

In [ ]:
# Train three-class classifier
history = model.fit(

    train_data,

    validation_data=val_data,

    epochs=10
)

In [ ]:
# Evaluate model performance
loss, accuracy = model.evaluate(val_data)

# Display accuracy
print("Validation Accuracy:", accuracy)

In [ ]:
# Predict validation images
predictions = model.predict(val_data)

# Convert probabilities into class labels
predicted_classes = np.argmax(
    predictions,
    axis=1
)

# Actual labels
true_classes = val_data.classes

# Retrieve class names
class_labels = list(
    val_data.class_indices.keys()
)

In [ ]:
# Display precision, recall and F1-score
print(classification_report(
    true_classes,
    predicted_classes,
    target_names=class_labels
))

In [ ]:
# Generate confusion matrix
cm = confusion_matrix(
    true_classes,
    predicted_classes
)

# Create figure
plt.figure(figsize=(8,6))

# Plot confusion matrix
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_labels,
    yticklabels=class_labels
)

# Labels
plt.xlabel("Predicted")
plt.ylabel("Actual")

# Title
plt.title("Three-Class Confusion Matrix")

# Display heatmap
plt.show()

In [ ]:
# Save three-class model
model.save(
    "../models/threeclass_papsmear_model.h5"
)